# เทรนโมเดล XGBoost พยากรณ์ราคาน้ำมันดิบ WTI (CL=F)
ข้อมูล: `crude_oil_15y.csv` (ราคาปิดรายวันย้อนหลัง 15 ปีจาก Yahoo Finance)

ขั้นตอน: ติดตั้ง → โหลดข้อมูล → ทำความสะอาด → สร้าง Feature → แบ่ง Train/Val/Test → เทรน XGBoost → วัดความแม่นยำ (MAE, RMSE, MAPE, Accuracy) เทียบกับ Baseline → ทดสอบพยากรณ์ล่วงหน้าหลายวัน → เทรนใหม่ด้วยข้อมูลทั้งหมด → พยากรณ์อนาคต → เซฟโมเดล

**กดรันทีละ cell จากบนลงล่าง** (Shift + Enter)

## 1. ติดตั้งไลบรารี (รันครั้งแรกครั้งเดียว)

In [ ]:
%pip install -q xgboost pandas numpy matplotlib scikit-learn

## 2. Import และตั้งค่า

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

CSV_PATH    = "crude_oil_15y.csv"
MODEL_DIR   = "MODEL"
MODEL_PATH  = os.path.join(MODEL_DIR, "oil_xgb_model.json")
PARAMS_PATH = os.path.join(MODEL_DIR, "oil_xgb_params.json")
FORECAST_PATH = os.path.join(MODEL_DIR, "oil_xgb_forecast.csv")

TEST_DAYS = 365     # ใช้ข้อมูล 1 ปีล่าสุดเป็นชุดทดสอบ (เท่ากับไฟล์ GRU)
VAL_RATIO = 0.15    # แบ่ง 15% ท้ายของ Train ไว้เป็น Validation (ใช้หยุดเทรนเมื่อไม่ดีขึ้น)
HORIZON   = 7       # พยากรณ์ล่วงหน้า 7 วันทำการ (ตามขอบเขตข้อ 2.3.6.2)
CLIP_SD   = 5       # ตัดผลตอบแทนรายวันที่เกิน ±5 เท่าของส่วนเบี่ยงเบน (กันวันผิดปกติ)
LAGS      = 20      # ใช้ผลตอบแทนย้อนหลัง 20 วันเป็น feature
SEED      = 42

os.makedirs(MODEL_DIR, exist_ok=True)
plt.rcParams["figure.figsize"] = (14, 5)
print("XGBoost", xgb.__version__)

## 3. โหลดข้อมูล
> ถ้าใช้ **Google Colab** ให้รัน cell ข้างล่างเพื่ออัปโหลดไฟล์ CSV (รันบนเครื่องตัวเองข้ามได้)

In [ ]:
try:
    from google.colab import files
    files.upload()
except ImportError:
    print("ไม่ได้รันบน Colab — ข้าม cell นี้ได้")

In [ ]:
raw = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
print("จำนวนแถว:", len(raw))
raw.head()

## 4. ทำความสะอาดข้อมูล

In [ ]:
df = raw.copy()

# 4.1 แปลงชนิดข้อมูล และเลือกคอลัมน์ที่ใช้
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
df = df[["Date", "Close"]]

# 4.2 ตัดแถวว่าง/วันที่ซ้ำ แล้วเรียงตามวันที่
before = len(df)
df = df.dropna().drop_duplicates(subset="Date", keep="last").sort_values("Date")
print("ตัดแถวว่าง/ซ้ำ:", before - len(df))

# 4.3 ตัดแถวของวันนี้ (ตลาดยังไม่ปิด)
today = pd.Timestamp.today().normalize()
print("ตัดแถววันนี้:", (df["Date"] >= today).sum())
df = df[df["Date"] < today]

# 4.4 ปัดทศนิยม 2 ตำแหน่ง
df["Close"] = df["Close"].round(2)

# 4.5 ราคาติดลบ/เป็นศูนย์ (20 เม.ย. 2020 WTI ติดลบ −37.63) คำนวณ log ไม่ได้ จึงตัดออก
bad = df["Close"] <= 0
print("ราคา <= 0 ที่ตัดออก:", bad.sum(), df.loc[bad, "Date"].dt.date.tolist())
df = df[~bad]

df = df.reset_index(drop=True)
print("เหลือ", len(df), "แถว ตั้งแต่", df["Date"].min().date(), "ถึง", df["Date"].max().date())

In [ ]:
plt.plot(df["Date"], df["Close"])
plt.title("WTI Crude Oil (CL=F) Close Price"); plt.grid(alpha=.3); plt.show()

## 5. แปลงเป็นผลตอบแทนรายวัน (log return) และตัดค่าสุดโต่ง
XGBoost (แบบต้นไม้) **พยากรณ์ค่าที่สูงกว่าที่เคยเห็นตอนเทรนไม่ได้** ถ้าใช้ราคาตรง ๆ จะพยากรณ์ราคาทอง/น้ำมันช่วงที่ขึ้นทำสถิติใหม่ไม่ได้เลย
จึงให้โมเดลพยากรณ์ **% การเปลี่ยนแปลงรายวัน** แล้วแปลงกลับเป็นราคา: `ราคาพรุ่งนี้ = ราคาวันนี้ × exp(ผลตอบแทนที่พยากรณ์)`

In [ ]:
df["ret"] = np.log(df["Close"]).diff()
df = df.dropna().reset_index(drop=True)

prices = df["Close"].values
dates  = df["Date"].values

test_start = int(np.searchsorted(dates, dates[-1] - np.timedelta64(TEST_DAYS, "D"), side="right"))
val_start  = int(test_start * (1 - VAL_RATIO))

sd0 = df["ret"].iloc[:val_start].std()       # ใช้สถิติจาก Train เท่านั้น
n_clip = (df["ret"].abs() > CLIP_SD * sd0).sum()
rets = df["ret"].clip(-CLIP_SD * sd0, CLIP_SD * sd0).values
print(f"ตัดค่าผลตอบแทนสุดโต่ง (> {CLIP_SD} SD): {n_clip} วัน")

## 6. สร้าง Feature
ทุก feature คำนวณจาก **ผลตอบแทนในอดีตเท่านั้น** (ไม่ใช้ข้อมูลของวันที่จะพยากรณ์) เพื่อไม่ให้ข้อมูลอนาคตรั่ว และเพื่อให้พยากรณ์ต่อกันหลายวันได้
- `lag_1 … lag_20` ผลตอบแทนย้อนหลัง 1–20 วัน
- `mean_5/10/20` ค่าเฉลี่ยผลตอบแทน (แนวโน้มระยะสั้น)
- `std_5/10/20` ความผันผวน
- `mom_20/60` ราคาปัจจุบันห่างจากค่าเฉลี่ย 20/60 วันกี่ % (โมเมนตัม)

In [ ]:
def make_features(r):
    # r = array ผลตอบแทนถึงวันที่ t-1 → คืน feature สำหรับพยากรณ์วันที่ t
    s = pd.Series(r)
    feats = {f"lag_{k}": s.shift(k - 1) for k in range(1, LAGS + 1)}
    for w in (5, 10, 20):
        feats[f"mean_{w}"] = s.rolling(w).mean()
        feats[f"std_{w}"]  = s.rolling(w).std()
    logp = s.cumsum()                                  # log ราคาสะสม (เริ่มที่ 0)
    for w in (20, 60):
        feats[f"mom_{w}"] = logp - logp.rolling(w).mean()
    return pd.DataFrame(feats)

F = make_features(rets).shift(1)     # feature ของวันที่ t ใช้ข้อมูลถึงวันที่ t-1
FEATURES = list(F.columns)
y = pd.Series(rets)                   # target = ผลตอบแทนของวันที่ t

valid = F.notna().all(axis=1).values
idx = np.arange(len(rets))
tr = valid & (idx < val_start)
va = valid & (idx >= val_start) & (idx < test_start)
te = valid & (idx >= test_start)

X_train, y_train = F[tr], y[tr]
X_val,   y_val   = F[va], y[va]
X_test,  y_test  = F[te], y[te]
i_test = idx[te]

print(f"Features: {len(FEATURES)}")
print(f"Train: {len(X_train)} แถว  ({str(dates[idx[tr][0]])[:10]} ถึง {str(dates[idx[tr][-1]])[:10]})")
print(f"Val  : {len(X_val)} แถว  ({str(dates[idx[va][0]])[:10]} ถึง {str(dates[idx[va][-1]])[:10]})")
print(f"Test : {len(X_test)} แถว  ({str(dates[i_test[0]])[:10]} ถึง {str(dates[i_test[-1]])[:10]})")

## 7. เทรนโมเดล XGBoost
ใช้ early stopping: หยุดเพิ่มต้นไม้เมื่อผลบน Validation ไม่ดีขึ้น 50 รอบติด

In [ ]:
XGB_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=3,            # ต้นไม้ตื้น กัน overfit (ข้อมูลการเงินมี noise สูง)
    min_child_weight=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5.0,
    objective="reg:squarederror",
    random_state=SEED,
)

model = xgb.XGBRegressor(**XGB_PARAMS, early_stopping_rounds=50, eval_metric="rmse")
model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=100)

BEST_N = model.best_iteration + 1
print("จำนวนต้นไม้ที่ดีที่สุด:", BEST_N)

In [ ]:
ev = model.evals_result()
plt.plot(ev["validation_0"]["rmse"], label="train rmse")
plt.plot(ev["validation_1"]["rmse"], label="val rmse")
plt.axvline(BEST_N - 1, color="gray", ls="--", label="best")
plt.title("Training history"); plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
imp.plot.barh(figsize=(8, 7), title="Feature importance"); plt.grid(alpha=.3); plt.show()

## 8. พยากรณ์ชุด Test (ล่วงหน้า 1 วัน)
ทุกวันในชุดทดสอบ ใช้ข้อมูลจริงถึงเมื่อวาน พยากรณ์ราคาวันนี้

In [ ]:
pred_ret = model.predict(X_test)
prev_price = prices[i_test - 1]
result = pd.DataFrame({
    "ds": dates[i_test],
    "y": prices[i_test],
    "yhat": prev_price * np.exp(pred_ret),
    "naive": prev_price,                       # Baseline: ราคาพรุ่งนี้ = ราคาวันนี้
})
result.head()

## 9. วัดความแม่นยำ: MAE, RMSE, MAPE, Accuracy
- **MAE** ผิดเฉลี่ยกี่ดอลลาร์ | **RMSE** ลงโทษวันที่ผิดมากหนักกว่า | **MAPE** ผิดเฉลี่ยกี่ %
- **Accuracy** = 100 − MAPE
- **Naive** = เดาว่าราคาพรุ่งนี้เท่ากับวันนี้ ถ้า XGBoost แพ้ Naive แปลว่าโมเดลยังไม่ได้ช่วยอะไร
- **Direction Accuracy** = ทายถูกกี่ % ว่าพรุ่งนี้ราคาขึ้นหรือลง (50% = เท่ากับโยนเหรียญ)

In [ ]:
def evaluate(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAPE (%)": mape,
            "Accuracy (%)": 100 - mape}

metrics = pd.DataFrame({"XGBoost": evaluate(result["y"], result["yhat"]),
                        "Naive (ราคาวันก่อน)": evaluate(result["y"], result["naive"])}).round(2)
print("พยากรณ์ล่วงหน้า 1 วัน (ชุดทดสอบ 1 ปี):")
print(metrics)

actual_up = result["y"].values > result["naive"].values
pred_up   = result["yhat"].values > result["naive"].values
dir_acc = (actual_up == pred_up).mean() * 100
print(f"\nDirection Accuracy (ทายขึ้น/ลงถูก): {dir_acc:.2f}%")
print(f">>> ความแม่นยำของโมเดล (Accuracy) = {metrics.loc['Accuracy (%)', 'XGBoost']:.2f}%")

In [ ]:
plt.plot(result["ds"], result["y"], label="Actual", color="black")
plt.plot(result["ds"], result["yhat"], label="XGBoost (1-day ahead)", color="tab:green", alpha=.8)
m = metrics["XGBoost"]
plt.title(f"Test 1-day ahead: MAE={m['MAE']:.2f}  RMSE={m['RMSE']:.2f}  MAPE={m['MAPE (%)']:.2f}%  Accuracy={m['Accuracy (%)']:.2f}%")
plt.legend(); plt.grid(alpha=.3); plt.show()

## 9.1 ทดสอบพยากรณ์ล่วงหน้า 7 วัน
เริ่มจากหลายจุดในชุดทดสอบ (ทุก ๆ 7 วัน) แล้วให้ XGBoost พยากรณ์ต่อกันเอง 7 วัน โดยไม่รู้ราคาจริง (recursive) แล้วเฉลี่ยค่า MAE, RMSE, MAPE

In [ ]:
def forecast_recursive(m, r_hist, last_price, steps):
    r = list(r_hist)
    out, price = [], last_price
    for _ in range(steps):
        x = make_features(np.array(r)).iloc[[-1]][FEATURES]   # feature ล่าสุด จากข้อมูลถึงวันก่อน
        p = float(m.predict(x)[0])
        r.append(p)
        price = price * np.exp(p)
        out.append(price)
    return np.array(out)

rows = []
for start in range(test_start, len(rets) - HORIZON + 1, HORIZON):
    y_true = prices[start:start + HORIZON]
    p_x  = forecast_recursive(model, rets[:start], prices[start - 1], HORIZON)
    p_nv = np.full(HORIZON, prices[start - 1])
    g, n = evaluate(y_true, p_x), evaluate(y_true, p_nv)
    rows.append({"start": str(dates[start])[:10], **{f"XGB {k}": v for k, v in g.items()}, **{f"Naive {k}": v for k, v in n.items()}})
    print(f"{str(dates[start])[:10]}  XGBoost MAPE {g['MAPE (%)']:6.2f}%   Naive MAPE {n['MAPE (%)']:6.2f}%")

ms = pd.DataFrame(rows)
keys = ["MAE", "RMSE", "MAPE (%)", "Accuracy (%)"]
metrics_h = pd.DataFrame({"XGBoost": [ms[f"XGB {k}"].mean() for k in keys],
                          "Naive (ราคาล่าสุด)": [ms[f"Naive {k}"].mean() for k in keys]}, index=keys).round(2)
print(f"\nค่าเฉลี่ยจาก {len(ms)} รอบ (พยากรณ์ล่วงหน้า {HORIZON} วัน):")
print(metrics_h)

## 10. เทรนใหม่ด้วยข้อมูลทั้งหมด แล้วพยากรณ์อนาคต
ใช้จำนวนต้นไม้เท่ากับที่ดีที่สุดจากข้อ 7

In [ ]:
final_model = xgb.XGBRegressor(**{**XGB_PARAMS, "n_estimators": BEST_N})
final_model.fit(F[valid], y[valid])

future_prices = forecast_recursive(final_model, rets, prices[-1], HORIZON)
future_dates  = pd.bdate_range(df["Date"].max() + pd.Timedelta(days=1), periods=HORIZON)
forecast = pd.DataFrame({"ds": future_dates, "yhat": future_prices.round(2)})
forecast.head(10)

In [ ]:
hist = df[df["Date"] >= df["Date"].max() - pd.Timedelta(days=365)]
plt.plot(hist["Date"], hist["Close"], color="black", label="Actual")
plt.plot(forecast["ds"], forecast["yhat"], color="tab:green", label="XGBoost forecast")
plt.title(f"XGBoost forecast next {HORIZON} business days"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 11. เซฟโมเดล ค่าที่ต้องใช้ และผลพยากรณ์

In [ ]:
final_model.save_model(MODEL_PATH)
with open(PARAMS_PATH, "w") as f:
    json.dump({"FEATURES": FEATURES, "LAGS": LAGS, "CLIP": float(CLIP_SD * sd0),
               "last_date": str(df["Date"].max().date()), "last_price": float(prices[-1])}, f, indent=2)

forecast.to_csv(FORECAST_PATH, index=False)
metrics.to_csv(os.path.join(MODEL_DIR, "oil_xgb_metrics_1day.csv"))
metrics_h.to_csv(os.path.join(MODEL_DIR, f"oil_xgb_metrics_{HORIZON}day.csv"))

print("เซฟโมเดล     ->", MODEL_PATH)
print("เซฟค่าที่ใช้   ->", PARAMS_PATH)
print("เซฟพยากรณ์   ->", FORECAST_PATH)

## 12. ทดสอบโหลดโมเดลกลับมาใช้

In [ ]:
loaded = xgb.XGBRegressor()
loaded.load_model(MODEL_PATH)
with open(PARAMS_PATH) as f:
    params = json.load(f)

check = forecast_recursive(loaded, rets, params["last_price"], 5)
print("พยากรณ์ 5 วันข้างหน้าจากโมเดลที่โหลด:", check.round(2))